In [10]:
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, conversion

# load R package
ro.r('.libPaths(c("/dcs/23/u2200504/R/x86_64-redhat-linux-gnu-library/4.5", .libPaths()))')
ro.r('library(dagbagM)')

In [11]:
#assigns node types. 'c' for continuous, 'b' for binary
def infer_type(df):
    import rpy2.robjects as ro
    node_types=[]
    for col in df.columns:
        x=df[col].dropna()
        unique_vals=x.unique()
        if np.issubdtype(x.dtype, np.number):
            if len(unique_vals) == 2 and set(unique_vals).issubset({0, 1}):
                node_types.append("b")
            else:
                node_types.append("c")
        else:
            if len(unique_vals) == 2:
                node_types.append("b")
            else:
                node_types.append("c")
    return ro.StrVector(node_types)

In [12]:
def run_dagbagm(df: pd.DataFrame, seed=1):
    df_clean = df.dropna().copy()
    #infer node types on transformed df
    node_type = infer_type(df_clean)

    print("Columns used in dagbagM:", df_clean.columns.tolist())
    print("dtypes:", df_clean.dtypes)

    with (ro.default_converter + pandas2ri.converter).context():
        #Python ->R
        Y_r = conversion.py2rpy(df_clean)

        ro.globalenv["Y"] = Y_r
        ro.globalenv["node_type"] = node_type
        ro.globalenv["seed"] = seed
        ro.r('''
        set.seed(seed)
        Y_df <- as.data.frame(Y)

        # Convert to numeric matrix for hc
        Y_mat <- as.matrix(Y_df)
        
        temp <- dagbagM::hc(
          Y = Y_mat,
          nodeType = node_type,
          whiteList = NULL,
          blackList = NULL,
          tol = 1e-6,
          standardize = FALSE,
          maxStep = 1000,
          restart = 10,
          verbose = FALSE
        )
        adj_mat <- temp$adjacency
        col_names <- colnames(Y_mat)
        ''')
        #R -> Python
        adjacency = conversion.rpy2py(ro.r('adj_mat'))
        col_names= list(ro.r('col_names'))

    return np.asarray(adjacency), col_names #list(df_clean.columns) #returns labels.

In [13]:
import networkx as nx
import matplotlib.pyplot as plt

def draw_graph(adj, nodes, output_path):
    G = nx.DiGraph()
    #add nodes
    G.add_nodes_from(nodes)
    
    #add directed edges, adj[i,j] = 1 => i -> j
    for i, src in enumerate(nodes):
        for j, tgt in enumerate(nodes):
            if adj[i, j] == 1:
                G.add_edge(src, tgt)

    plt.figure(figsize=(10,8))
    pos = nx.spring_layout(G, k=1.2, iterations=500, seed=0)
    nx.draw(G, pos, with_labels=True, labels={node: node for node in nodes},
            node_size=900, font_size=8, arrowsize=10)
    plt.savefig(out_path, dpi=150)
    plt.close()

In [14]:
from graphviz import Digraph
from pathlib import Path
import numpy as np

def draw_graphviz_dag(adj, out_path, node_labels=None, engine="dot"):

    adj = np.asarray(adj)
    if adj.ndim == 1:
        adj = adj.reshape(1, 1)

    n = adj.shape[0]

    # Default labels
    if node_labels is None:
        node_labels = [f"X{i}" for i in range(n)]
    else:
        node_labels = list(node_labels)[:n]

    out_path = Path(out_path)
    g = Digraph(format="png", engine=engine)

    # Thesis‑friendly black & white style
    g.attr(rankdir="TB")  # top‑to‑bottom; use "LR" if you prefer left‑to‑right
    g.attr(
        "node",
        shape="ellipse",
        style="solid",
        color="black",
        fontname="Helvetica",   # or "Times New Roman" / "Palatino"
        fontsize="10",
    )
    g.attr(
        "edge",
        color="black",
        arrowsize="0.7",
    )

    # Add nodes
    for name in node_labels:
        g.node(name, label=name)

    # Add edges for weights above threshold
    for i, src in enumerate(node_labels):
        for j, tgt in enumerate(node_labels):
            w = adj[i, j]
            if abs(w) > 0:
                g.edge(src, tgt)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    g.render(filename=out_path.with_suffix("").as_posix(), cleanup=True)


In [15]:
#scoring utility function given two adjacency matricies
def score_graph(W_est, W_true, labels_est, labels_true):
    #number of variables
    p = W_est.shape[0]

    #node orderings
    labels_est = list(labels_est)
    labels_true = list(labels_true)
    common = sorted(set(labels_est) & set(labels_true))

    #map ids
    idx_est = [labels_est.index(v) for v in common]
    idx_true = [labels_true.index(v) for v in common]

    W_est_aligned = np.asarray(W_est)[np.ix_(idx_est, idx_est)]
    W_true_aligned = np.asarray(W_true)[np.ix_(idx_true, idx_true)]

    est = (W_est!=0).astype(int)
    true = (W_true !=0).astype(int)

    #true positive, false positive, false negative, true negative
    tp = np.sum((est==1) & (true==1))
    fp = np.sum((est==1) & (true == 0))
    fn = np.sum((est==0) & (true == 1))
    tn = np.sum((est==0) & (true == 0))

    #structural hamming distance, true positive rate, false positive rate
    shd = fp + fn
    tpr = tp/(tp+fn) if (tp+fn) > 0 else np.nan
    fpr = fp/(fp+tn) if (fp + tn)>0 else np.nan
    fdr = fp/(tp+fp) if (tp+fp)>0 else np.nan

    #store scores in dictionary format for each 'experiment'
    return dict(TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn), SHD=int(shd), TPR = tpr, FPR = fpr, FDR= fdr)

In [16]:
import os
from pathlib import Path
#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set causal pitfalls root
cp_root = project_root / "data" / "raw" / "CausalPitfallsData"
#test it works
cp_root

#path to put results
output_dir = project_root/"results"/"graphs_DAGBagM"
output_dir.mkdir(parents=True,exist_ok=True)

In [21]:
csv_files = sorted(cp_root.rglob("*.csv"))
results = []

for csv_path in csv_files:
    #relative path
    rel = csv_path.relative_to(cp_root)

    #we only examine those with known ground truths i.e with _truth
    if csv_path.stem.endswith("_truth"):
        continue

    #expected ground truth path
    truth_path = csv_path.with_name(csv_path.stem + "_truth.csv")
    if not truth_path.exists():
        print("  Skipped (no ground-truth file)")
        continue

    print(f"Processing: {rel}")

    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            print("  Skipped (empty file)")
            continue

        # Run DAGBagM on this dataset
        adj,nodes = run_dagbagm(df, seed=1)

        #Output schema scenario__file__DAGBagM.png
        parts = rel.parts           
        scenario = parts[0] if len(parts) > 1 else "root"
        name_no_ext = csv_path.stem

        out_name = f"{scenario}__{name_no_ext}__DAGBagM.png"
        out_path = output_dir / out_name

        # Save PNG
        draw_graphviz_dag(adj, out_path,nodes)
        print(f"  Saved graph to {out_path.relative_to(project_root)}")

        #load ground truth and score
        gt_df = pd.read_csv(truth_path)
        gt_df.index.name = None
        gt_df.columns.name = None

        W_true = gt_df.to_numpy()
            
        # compute scores
        metrics = score_graph(adj, W_true, labels_est=list(df.columns), labels_true=list(gt_df.columns))
        print("metrics computed")
        print(metrics)
        metrics.update(
            dict(
                scenario=scenario,
                dataset=name_no_ext,
                algo="dagbagm"
            )
        )
        results.append(metrics)
        print(f"  SHD={metrics['SHD']}, TPR={metrics['TPR']:.3f}, FDR={metrics['FDR']:.3f}")

    except Exception as e:
        print(f"  ERROR on {rel}: {e}")

scores_df = pd.DataFrame(results)

  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
Processing: casual_effect/device_failure_data.csv
Columns used in dagbagM: ['usage_hours', 'baseline_error', 'patch_v1', 'post_patch_error', 'patch_v2', 'failure', 'random_noise']
dtypes: usage_hours           int64
baseline_error        int64
patch_v1              int64
post_patch_error      int64
patch_v2              int64
failure               int64
random_noise        float64
dtype: object
  Saved graph to results/graphs_DAGBagM/casual_effect__device_failure_data__DAGBagM.png
metrics computed
{'

In [22]:
scores_df

,TP,FP,FN,TN,SHD,TPR,FPR,FDR,scenario,dataset,algo
0,2,8,7,32,15,0.222222,0.200000,0.800000,casual_effect,device_failure_data,dagbagm
1,3,4,6,36,10,0.333333,0.100000,0.571429,casual_effect,student_tutoring_data,dagbagm
2,2,2,1,11,3,0.666667,0.153846,0.500000,causal_direction_iv,clinical_trial_sem,dagbagm
3,3,1,0,12,1,1.000000,0.076923,0.250000,causal_direction_iv,ecommerce_sem,dagbagm
4,2,2,1,11,3,0.666667,0.153846,0.500000,causal_direction_iv,environment_sem,dagbagm
5,2,2,1,11,3,0.666667,0.153846,0.500000,causal_direction_iv,marketing_sem,dagbagm
6,1,4,3,8,7,0.250000,0.333333,0.800000,counterfactual_reasoning,climate_impact_sem,dagbagm
7,1,4,3,8,7,0.250000,0.333333,0.800000,counterfactual_reasoning,clinical_trial_sem,dagbagm
8,1,4,3,8,7,0.250000,0.333333,0.800000,counterfactual_reasoning,education_performance_sem,dagbagm
9,1,4,3,8,7,0.250000,0.333333,0.800000,counterfactual_reasoning,investment_outcome_sem,dagbagm


In [23]:
scores_path = project_root / "results" / "scores"/"scores_dagbagm.csv"
scores_path.parent.mkdir(parents=True, exist_ok=True)
scores_df.to_csv(scores_path, index=False)